In [2]:
### USE THIS SCRIPT TO CREATE A NEW JOBLIB FILE FOR THE MODEL




# ============================================================
# Core Python
# ============================================================
from datetime import datetime, timedelta, date, time
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================
# Numerical / Data
# ============================================================
import numpy as np
import pandas as pd
import os
import duckdb
DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb"
con = duckdb.connect(DB_PATH) 
print("Connected:", DB_PATH)

# ============================================================
# Machine Learning – sklearn
# ============================================================
# Models

# Preprocessing
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import LabelEncoder, StandardScaler

# Model selection
from sklearn.model_selection import train_test_split, GridSearchCV

# Evaluation / inspection
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
)
from sklearn.inspection import permutation_importance

# Feature selection
from sklearn.feature_selection import SelectKBest, RFE, mutual_info_classif
from sklearn.metrics import brier_score_loss
# ============================================================
# XGBoost
# ============================================================
import xgboost as xgb
from xgboost import XGBClassifier

# ============================================================
# Model persistence
# ============================================================
from pathlib import Path

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

from joblib import dump, load

print("✅ Loaded pipeline + feature list")

print("✅ Playoff model loaded")



# ============================================================
# Explainability
# ============================================================
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️ SHAP not installed — using permutation importance fallback.")

# ============================================================
# Visualization
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("models", exist_ok=True)


# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


Connected: C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb
✅ Loaded pipeline + feature list
✅ Playoff model loaded


## Train DF

In [3]:
train_df = con.execute('select* from team_daily_stats').df()
train_df = train_df[train_df["season_id"] != 20252026]
train_df = train_df.sort_values(by='game_id').reset_index(drop=True)
train_df.head()

,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,ppng_rank,c_pp_ga,c_times_sh,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs,goals_for,goals_against,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct
0,20132014,2013020001,2013-10-01,10,TOR,Toronto Maple Leafs,1,<NA>,0,NaN,NaN,0.496280,7,7.0,700.0,1,1.0,1,1.0,0.14285,0.142857,0,0.0,4,4.0,1.250000,1.00000,1,1.0,405.0,0,0.0,1,1.0,7.0,1.0,4,0.0,4.0,1,0.91891,3.01003,9,7,2,9,7,2,9.0,7.0,2.0,63,68,-5,45,53,-8,-5,1,63.0,68.0,-5.0,31,38,4,34,6,25,32,16,22,34,0.272727,0.727273,0.531915,0.468085,0.484848,0.515152,31,38,4,65,31.0,34.0,7,6,27,25,28,32,38,5,15,18,1,2,1,10.526316,37,11,9,18,91.891892,102.418208,4,5,15,18,1,2,1,3,11,9,18,0.105263,0.918919,102.418208,63,68,45,53,-8,-8.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,4.0,1,1.0,W,1.0,1,1,0,0,2,4,3,1,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,4,3,1,Atlantic,1,7,1,1,Eastern,1,14,1,1,2,25,1,2,0,0,4,3,0,0,0.0,0.496280,2.765947,2,4.00000
1,20132014,2013020001,2013-10-01,8,MTL,Montréal Canadiens,0,<NA>,0,NaN,NaN,0.486969,4,4.0,405.0,0,0.0,-1,-1.0,0.00000,-0.250000,1,1.0,7,7.0,0.857142,0.85714,-1,-1.0,700.0,1,1.0,0,0.0,4.0,-1.0,6,1.0,7.0,2,0.89473,4.06549,7,9,-2,7,9,-2,7.0,9.0,-2.0,68,63,5,53,45,8,5,1,68.0,63.0,5.0,34,37,3,31,16,22,34,6,25,32,0.727273,0.272727,0.468085,0.531915,0.515152,0.484848,34,37,3,65,34.0,31.0,15,16,20,22,38,34,37,11,9,18,3,0,0,8.108108,38,5,15,18,89.473684,97.581792,3,11,9,18,3,0,0,4,5,15,18,0.081081,0.894737,97.581792,68,63,53,45,8,8.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,3.0,-1,-1.0,L,1.0,1,0,1,0,0,3,4,-1,0,0,1,0,1,0,0,0,0,3,4,-1,0,0,0,0,0,0,0,0,0,0,Atlantic,8,8,6,8,Eastern,15,16,8,15,29,30,17,29,9,1,3,4,0,0,0.0,0.000000,3.637516,2,5.99998
2,20132014,2013020002,2013-10-01,15,WSH,Washington Capitals,0,<NA>,0,NaN,NaN,0.480114,6,6.0,388.0,3,3.0,3,3.0,0.50000,0.500000,0,0.0,4,4.0,0.750000,0.75000,-1,-1.0,387.0,1,1.0,0,0.0,6.0,3.0,1,1.0,4.0,3,0.85294,5.06329,4,7,-3,4,7,-3,4.0,7.0,-3.0,54,58,-4,39,38,1,-4,1,54.0,58.0,-4.0,30,32,4,34,11,21,24,15,22,24

In [4]:
# -------------------------------------------------
# Normalize made_playoffs: once 1 appears → always 1
# -------------------------------------------------
final_outcomes = (
    train_df
    .groupby(["season_id", "tri_code"])["made_playoffs"]
    .apply(lambda s: 1 if (s == 1).any() else 0)
    .reset_index()
)

train_df = (
    train_df
    .drop(columns=["made_playoffs"])
    .merge(final_outcomes, on=["season_id", "tri_code"], how="left")
)

# -------------------------------------------------
# Assertion: constant per team-season
# -------------------------------------------------
assert (
    train_df
    .groupby(["season_id", "tri_code"])["made_playoffs"]
    .nunique()
    .le(1)
    .all()
), "❌ made_playoffs varies within a team-season"


train_df = train_df.sort_values(by=['season_id','tri_code', 'game_id'])


In [5]:

conference_sizes = (
    train_df.groupby(["season_id", "conference"])["tri_code"]
    .nunique()
    .reset_index()
    .rename(columns={"tri_code": "conference_team_count"})
)

train_df = train_df.merge(conference_sizes, on=["season_id", "conference"], how="left")

train_df["conf_playoff_chance_baseline"] = 8 / train_df["conference_team_count"]

train_df["streak_direction"] = train_df["streak"].map({
    "W": 1,
    "L": -1
}).fillna(0).astype(int)


In [6]:


train_df["streak_signed"] = (
    train_df["streak_direction"] *
    train_df["streak_count"].fillna(0).astype(int)
)


train_df["streak_signed"] = train_df["streak_signed"].clip(-7, 7)


train_df.drop(columns=["streak", "streak_direction"], inplace=True)


assert (
    train_df
    .groupby(["season_id", "tri_code", "game_date"])["streak_signed"]
    .nunique()
    .le(1)
    .all()
), "❌ streak_signed varies within a team-day"

print("✅ streak_signed feature created successfully")


✅ streak_signed feature created successfully


In [7]:

def bucket_rank(rank):
    if rank <= 3:
        return "tier1_elite"
    elif rank <= 6:
        return "tier2_strong"
    elif rank <= 10:
        return "tier3_bubble"
    else:
        return "tier4_bottom"
        
train_df["rank_bucket"] = train_df["conference_rank"].apply(bucket_rank)


In [8]:

# =====================================================
# DROP HIGH-NOISE / HIGH-RISK FEATURES (MODEL ONLY)
# =====================================================

DROP_COLS = [

    # -------------------------------------------------
    # 1. SINGLE-GAME RAW STATS (NOISY)
    # -------------------------------------------------
    "shots_for",
    "shots_against",
    "goals_for",
    "goals_against",

    "hd_sog",
    "md_sog",
    "ld_sog",

    "hd_goals",
    "md_goals",
    "ld_goals",

    "hd_against",
    "md_against",
    "ld_against",

    "game_savePct",
    "game_GAA",

    # -------------------------------------------------
    # 2. RAW PER-GAME SPECIAL TEAMS COUNTS
    # (keep cumulative / per-game rates instead)
    # -------------------------------------------------
    "pp_gf",
    "pp_ga",
    "sh_gf",
    "sh_ga",
    "times_sh",

    # -------------------------------------------------
    # 3. REDUNDANT / DOMINANT RANK FEATURES
    # (keep rank_bucket + conference_rank only)
    # -------------------------------------------------

    "division_home_rank",
    "division_road_rank",
    "division_last_10_rank",

    "conference_home_rank",
    "conference_road_rank",
    "conference_last_10_rank",

    "league_rank",
    "league_home_rank",
    "league_road_rank",
    "league_last_10_rank",



    # -------------------------------------------------
    # 4. OPTIONAL: STRONG LATE-SEASON PROXIES
    # (uncomment if probabilities converge too fast)
    # -------------------------------------------------
    # "in_playoff_spot",
]

# =====================================================
# APPLY DROP (SAFE — ignores missing)
# =====================================================

print(f"Dropping {len(DROP_COLS)} high-risk features...")

train_df_model = train_df.drop(columns=DROP_COLS, errors="ignore")

print("Remaining feature count:", train_df_model.shape[1])

train_df_model.head()


Dropping 30 high-risk features...
Remaining feature count: 171


,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga_pg,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi,pp_ga_pg,sh_gf_pg,c_pp_opps,c_pp_net_g,ppng_rank,c_pp_ga,c_times_sh,pk_rank,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shot_pct,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,conference,conference_rank,wildcard_rank,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct,made_playoffs,conference_team_count,conf_playoff_chance_baseline,streak_signed,rank_bucket
0,20132014,2013020006,2013-10-02,24,ANA,Anaheim Ducks,0,<NA>,0,NaN,NaN,0.494599,2,2.000000,240.0,0.0,0,0.0,0.00000,0.000000,0.0,4.0,0.500000,0.50000,-2,-2.000000,369.0,2.000000,0.0,2.0,0.0,6,2.0,4.0,6,2,2,0,2,2,0,2.0,2.000000,0.000000,55,54,1,43,40,3,1,1,55.000000,54.000000,1.00,23,36,1,28,8,16,32,8,21,26,0.500000,0.500000,0.432432,0.567568,0.551724,0.448276,23,36,1,51,23.000000,28.00,10,8,19,16,31,32,2.777778,79.310345,82.088123,1,8,8,21,1,0,0,6,4,9,16,0.027778,0.793103,82.088123,55,54,43,40,3,3.000000,81,0,0.000000,1,0,0,0.000000,0,0,0,0,0.000000,0,0,1.000000,-5,-5.00,1.0,1,0,1,0,0,1,6,-5,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,6,-5,Pacific,7,Western,14,8,0,0,0.00,0.000000,4.758600,0,2.00000,1,14.0,0.571429,-1,tier4_bottom
1,20132014,2013020029,2013-10-05,24,ANA,Anaheim Ducks,1,0,1,0.000000,0.000000,0.494599,3,1.500000,348.0,0.5,1,0.5,0.33333,0.333333,0.0,2.0,0.500000,0.50000,-2,-1.000000,303.0,1.000000,0.0,5.0,1.0,9,4.0,8.0,15,4,4,0,6,6,0,3.0,3.000000,0.000000,55,54,1,31,47,-16,2,2,55.000000,54.000000,1.00,28,30,4,27,9,19,27,13,15,27,0.409091,0.590909,0.558824,0.441176,0.500000,0.500000,51,66,5,106,25.500000,27.50,17,17,34,35,53,59,13.333333,90.909091,104.242424,5,13,18,36,4,1,0,9,14,17,31,0.075758,0.854839,93.059629,110,108,74,87,-13,-6.500000,80,1,0.500000,1,0,1,0.500000,0,0,0,2,0.500000,82,0,2.500000,-4,-2.00,1.0,2,1,1,0,2,5,9,-4,1,0,0,0,0,0,0,0,0,0,0,0,2,1,1,0,1,0,2,5,9,-4,Pacific,5,Western,10,4,1,1,0.50,0.247299,2.520707,0,2.00000,1,14.0,0.571429,1,tier3_bubble
2,20132014,2013020033,2013-10-06,24,ANA,Anaheim Ducks,1,1,2,0.500000,0.500000,0.500781,4,1.333333,480.0,0.0,0,0.0,0.00000,0.000000,0.0,1.0,0.666666,0.66666,-1,-0.333333,251.0,0.333333,0.0,9.0,1.0,13,5.0,11.0,15,6,5,1,12,11,1,4.0,3.666667,0.333333,65,40,25,48,27,21,27,3,58.333333,49.333333,9.00,40,34,3,23,15,28,25,8,16,17,0.652174,0.347826,0.636364,0.363636,0.595238,0.404762,91,100,8,169,30.333333,26.00,31,32,56,63,80,84,8.823529,88.235294,97.058824,8,23,32,48,7,1,0,11,17,22,40,0.080000,0.860759,94.075949,175,148,122,114,8,2.666667,79,2,0.666667,1,0,2,0.666667,1,0,0,4,0.666667,109,0,2.

## Test DF

In [9]:
test_df = con.execute('select* from team_daily_stats').df()
test_df = test_df[test_df["season_id"] == 20252026]
test_df = test_df.sort_values(by='game_id').reset_index(drop=True)
test_df.head()

,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,ppng_rank,c_pp_ga,c_times_sh,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,made_playoffs,goals_for,goals_against,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct
0,20252026,2025020001,2025-10-07,13,FLA,Florida Panthers,1,<NA>,0,NaN,NaN,0.497126,2,2.0,222.0,1,1.0,1,1.0,0.50000,0.500000,0,0.0,3,3.0,1.00,1.00,0,0.0,360.0,0,0.0,0,0.0,2.0,1.0,1,0.0,3.0,1,0.89473,2.00000,12,6,6,12,6,6,12.0,6.0,6.0,68,51,17,52,34,18,17,1,68.0,51.0,17.0,37,37,3,28,20,17,31,8,21,23,0.714286,0.285714,0.447368,0.552632,0.574074,0.425926,37,37,3,65,37.0,28.0,17,20,24,17,28,31,37,10,11,16,2,0,1,8.108108,19,3,5,12,89.473684,97.581792,3,10,11,16,2,0,1,2,3,5,12,0.081081,0.894737,97.581792,68,51,52,34,18,18.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,3.0,1,1.0,W,1.0,1,1,0,0,2,3,2,1,1,1,1,1,0,0,1,1,2,3,2,1,0,0,0,0,0,0,0,0,0,0,Atlantic,1,1,4,1,Eastern,2,1,7,2,3,1,12,3,0,3,3,2,0,0,0.497126,1.789460,6.000000,3,NaN
1,20252026,2025020001,2025-10-07,16,CHI,Chicago Blackhawks,0,<NA>,0,NaN,NaN,0.497453,3,3.0,360.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,2,2.0,0.50,0.50,-1,-1.0,222.0,1,1.0,0,0.0,3.0,0.0,4,1.0,2.0,4,0.91891,3.04740,2,8,-6,2,8,-6,2.0,8.0,-6.0,51,68,-17,34,52,-18,-17,1,51.0,68.0,-17.0,28,19,2,37,8,21,23,20,17,31,0.285714,0.714286,0.552632,0.447368,0.425926,0.574074,28,19,2,65,28.0,37.0,11,8,14,21,26,23,19,3,5,12,0,2,0,10.526316,37,10,11,16,91.891892,102.418208,2,3,5,12,0,2,0,3,10,11,16,0.105263,0.918919,102.418208,51,68,34,52,-18,-18.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,2.0,-1,-1.0,L,1.0,1,0,1,0,0,2,3,-1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,2,3,-1,Central,8,1,8,8,Western,15,3,16,15,30,7,32,30,9,2,2,3,0,0,0.000000,2.800286,6.000000,1,NaN
2,20252026,2025020002,2025-10-07,5,PIT,Pittsburgh Penguins,1,<NA>,0,NaN,NaN,0.481682,1,1.0,120.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,2,2.0,1.00,1.00,0,0.0,240.0,0,0.0,0,0.0,1.0,0.0,3,0.0,2.0,1,1.00000,0.00000,9,7,2,9,7,2,9.0,7.0,2.0,63,58,5,40,36,4,5,1,63.0,58.0,5.0,32,31,3,34,16,17,31,16,19,24,0.500000,0.500000,0.

In [10]:

conference_sizes = (
    test_df.groupby(["season_id", "conference"])["tri_code"]
    .nunique()
    .reset_index()
    .rename(columns={"tri_code": "conference_team_count"})
)

test_df = test_df.merge(conference_sizes, on=["season_id", "conference"], how="left")

test_df["conf_playoff_chance_baseline"] = 8 / test_df["conference_team_count"]
test_df["streak_direction"] = test_df["streak"].map({
    "W": 1,
    "L": -1
}).fillna(0).astype(int)


test_df["streak_signed"] = (
    test_df["streak_direction"] *
    test_df["streak_count"].fillna(0).astype(int)
)


test_df["streak_signed"] = test_df["streak_signed"].clip(-7, 7)


test_df.drop(columns=["streak", "streak_direction"], inplace=True)


assert (
    test_df
    .groupby(["season_id", "tri_code", "game_date"])["streak_signed"]
    .nunique()
    .le(1)
    .all()
), "❌ streak_signed varies within a team-day"

print("✅ streak_signed feature created successfully")
def bucket_rank(rank):
    if rank <= 3:
        return "tier1_elite"
    elif rank <= 6:
        return "tier2_strong"
    elif rank <= 10:
        return "tier3_bubble"
    else:
        return "tier4_bottom"
        
test_df["rank_bucket"] = test_df["conference_rank"].apply(bucket_rank)
# =====================================================
# DROP HIGH-NOISE / HIGH-RISK FEATURES (MODEL ONLY)
# =====================================================

DROP_COLS = [

    # -------------------------------------------------
    # 1. SINGLE-GAME RAW STATS (NOISY)
    # -------------------------------------------------
    "shots_for",
    "shots_against",
    "goals_for",
    "goals_against",

    "hd_sog",
    "md_sog",
    "ld_sog",

    "hd_goals",
    "md_goals",
    "ld_goals",

    "hd_against",
    "md_against",
    "ld_against",

    "game_savePct",
    "game_GAA",

    # -------------------------------------------------
    # 2. RAW PER-GAME SPECIAL TEAMS COUNTS
    # (keep cumulative / per-game rates instead)
    # -------------------------------------------------
    "pp_gf",
    "pp_ga",
    "sh_gf",
    "sh_ga",
    "times_sh",

    # -------------------------------------------------
    # 3. REDUNDANT / DOMINANT RANK FEATURES
    # (keep rank_bucket + conference_rank only)
    # -------------------------------------------------

    "division_home_rank",
    "division_road_rank",
    "division_last_10_rank",

    "conference_home_rank",
    "conference_road_rank",
    "conference_last_10_rank",

    "league_rank",
    "league_home_rank",
    "league_road_rank",
    "league_last_10_rank",



    # -------------------------------------------------
    # 4. OPTIONAL: STRONG LATE-SEASON PROXIES
    # (uncomment if probabilities converge too fast)
    # -------------------------------------------------
    # "in_playoff_spot",
]

# =====================================================
# APPLY DROP (SAFE — ignores missing)
# =====================================================

print(f"Dropping {len(DROP_COLS)} high-risk features...")

current_df = test_df.drop(columns=DROP_COLS, errors="ignore")

print("Remaining feature count:", current_df.shape[1])

current_df.head()


✅ streak_signed feature created successfully
Dropping 30 high-risk features...
Remaining feature count: 171


,season_id,game_id,game_date,team_id,tri_code,team_name,game_win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga_pg,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi,pp_ga_pg,sh_gf_pg,c_pp_opps,c_pp_net_g,ppng_rank,c_pp_ga,c_times_sh,pk_rank,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shot_pct,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,conference,conference_rank,wildcard_rank,made_playoffs,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct,conference_team_count,conf_playoff_chance_baseline,streak_signed,rank_bucket
0,20252026,2025020001,2025-10-07,13,FLA,Florida Panthers,1,<NA>,0,NaN,NaN,0.497126,2,2.0,222.0,1.0,1,1.0,0.50000,0.500000,0.0,3.0,1.00,1.00,0,0.0,360.0,0.0,0.0,2.0,1.0,1,0.0,3.0,1,12,6,6,12,6,6,12.0,6.0,6.0,68,51,17,52,34,18,17,1,68.0,51.0,17.0,37,37,3,28,20,17,31,8,21,23,0.714286,0.285714,0.447368,0.552632,0.574074,0.425926,37,37,3,65,37.0,28.0,17,20,24,17,28,31,8.108108,89.473684,97.581792,3,10,11,16,2,0,1,2,3,5,12,0.081081,0.894737,97.581792,68,51,52,34,18,18.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,3.0,1,1.0,1.0,1,1,0,0,2,3,2,1,1,1,1,1,0,0,1,1,2,3,2,1,0,0,0,0,0,0,0,0,0,0,Atlantic,1,Eastern,2,0,3,0,0,0.497126,1.789460,6.000000,3,NaN,16,0.5,1,tier1_elite
1,20252026,2025020001,2025-10-07,16,CHI,Chicago Blackhawks,0,<NA>,0,NaN,NaN,0.497453,3,3.0,360.0,0.0,0,0.0,0.00000,0.000000,0.0,2.0,0.50,0.50,-1,-1.0,222.0,1.0,0.0,3.0,0.0,4,1.0,2.0,4,2,8,-6,2,8,-6,2.0,8.0,-6.0,51,68,-17,34,52,-18,-17,1,51.0,68.0,-17.0,28,19,2,37,8,21,23,20,17,31,0.285714,0.714286,0.552632,0.447368,0.425926,0.574074,28,19,2,65,28.0,37.0,11,8,14,21,26,23,10.526316,91.891892,102.418208,2,3,5,12,0,2,0,3,10,11,16,0.105263,0.918919,102.418208,51,68,34,52,-18,-18.0,81,0,0.0,1,0,0,0.0,0,0,0,0,0.0,0,0,2.0,-1,-1.0,1.0,1,0,1,0,0,2,3,-1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,2,3,-1,Central,8,Western,15,9,2,0,0,0.000000,2.800286,6.000000,1,NaN,16,0.5,-1,tier4_bottom
2,20252026,2025020002,2025-10-07,5,PIT,Pittsburgh Penguins,1,<NA>,0,NaN,NaN,0.481682,1,1.0,120.0,0.0,0,0.0,0.00000,0.000000,0.0,2.0,1.00,1.00,0,0.0,240.0,0.0,0.0,1.0,0.0,3,0.0,2.0,1,9,7,2,9,7,2,9.0,7.0,2.0,63,58,5,40,36,4,5,1,63.0,58.0,5.0,32,31,3,34,16,17,31,16,19,24,0.500000,0.500000,0.472222,0.527778,0.563636,0.436364,19,31,3,37,32.0,34.0,11,11,8,8,22,22,9.677419,100.000000,109.677419,3,10,7,14,2,0,1,0,4,8,14,0.096774,1.000000,109.677419,63,58,40,36,4,4.0,81,1,1.0,0,0,1,1.0,1,0,0,2,1.0,164,0,3.0,3,3.0,1.0,1,1,0,0,2,3,0,3,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,1,2,3,0,3,Metropolitan,1,Eastern,1,0,<NA>,0,0,0.000000,0.481682,0.000000,6,2.0,16,0.5,1,tier1_elite
3,20252026,2025020002,2025-10-07,3,NYR,New York Rangers,0,<NA>,0,NaN,NaN,0.5066

## Build Model

In [11]:

def add_wc_field_pressure_features(df):
    """
    Add wild-card cutline and field-pressure features
    that respect NHL playoff structure:
      - Top 3 per division
      - Next 2 wild cards per conference
    """

    out = df.copy()

    # -------------------------------------------------
    # 1. Compute WC-eligible teams
    # -------------------------------------------------
    wc_eligible = out[out["division_rank"] > 3].copy()

    wc_cutline = (
        wc_eligible
        .sort_values(
            ["season_id", "game_date", "conference", "points", "points_pct"],
            ascending=[True, True, True, False, False]
        )
        .groupby(
            ["season_id", "game_date", "conference"],
            as_index=False
        )
        .nth(1)  # 2nd wild card
        .reset_index(drop=True)
        [["season_id", "game_date", "conference", "points", "points_pct"]]
        .rename(columns={
            "points": "wc2_points",
            "points_pct": "wc2_points_pct"
        })
    )

    # -------------------------------------------------
    # 2. Merge WC cutline back
    # -------------------------------------------------
    out = out.merge(
        wc_cutline,
        on=["season_id", "game_date", "conference"],
        how="left"
    )

    # -------------------------------------------------
    # 3. Margin to WC cutline
    # -------------------------------------------------
    out["wc_points_margin"] = out["points"] - out["wc2_points"]
    out["wc_points_pct_margin"] = out["points_pct"] - out["wc2_points_pct"]

    # -------------------------------------------------
    # 4. Binary status flags
    # -------------------------------------------------
    out["in_wc_spot"] = (out["wildcard_rank"] <= 2).astype(int)
    out["in_div_top3"] = (out["division_rank"] <= 3).astype(int)

    # -------------------------------------------------
    # 5. Bubble indicator
    # -------------------------------------------------
    out["on_wc_bubble"] = (
        out["wildcard_rank"].between(1, 4)
    ).astype(int)

    return out


train_df = add_wc_field_pressure_features(train_df)
test_df = add_wc_field_pressure_features(test_df)


In [12]:

def add_wc_density_features(df, windows=(2, 4, 6)):
    """
    Add wild-card crowding features:
    number of WC-eligible teams within X points of the WC cutline.
    """

    out = df.copy()

    # Only WC-eligible teams create WC pressure
    wc_mask = out["division_rank"] > 3

    for w in windows:
        col = f"wc_teams_within_{w}pts"

        out[col] = (
            out
            .assign(
                diff=lambda x: (x["points"] - x["wc2_points"]).abs()
            )
            .where(wc_mask)
            .groupby(["season_id", "game_date", "conference"])["diff"]
            .transform(lambda s: (s <= w).sum())
            .fillna(0)
            .astype(int)
        )

    return out

train_df = add_wc_density_features(train_df)
test_df = add_wc_density_features(test_df)
new_features = [
    "wc_points_margin",
    "wc_points_pct_margin",
    "in_wc_spot",
    "in_div_top3",
    "on_wc_bubble",
    "wc_teams_within_2pts",
    "wc_teams_within_4pts",
    "wc_teams_within_6pts",
]

missing = [c for c in new_features if c not in train_df.columns]
assert not missing, f"❌ Missing new features: {missing}"

print("✅ Field-pressure features present")


✅ Field-pressure features present


In [13]:

# =====================================================
# 1. SORT (SANITY)
# =====================================================

train_df_model = (
    train_df_model
    .sort_values(["season_id", "team_id", "game_date"])
    .reset_index(drop=True)
)

# =====================================================
# 2. DEFINE LEAKAGE + ID COLUMNS
# =====================================================

LEAK_COLS = [
    "game_id", "game_date", "date",
    "clinch_type", "made_playoffs",
    "team_name", "opponent",
    "season_id"
]

ID_COLS = [
    "team_id", "opponent_id", "tri_code"
]

# =====================================================
# 3. BUILD FEATURE MATRIX (SINGLE SOURCE OF TRUTH)
# =====================================================

X = (
    train_df_model
    .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
    .select_dtypes(include="number")
)

X = X.loc[:, X.isna().mean() < 0.20]
X = X.fillna(0)

y = train_df_model["made_playoffs"]

assert y.notna().all(), "❌ Target contains NaN values"

# =====================================================
# 4. CATEGORICAL DUMMIES (LOCKED)
# =====================================================

rank_categories = [
    "tier1_elite",
    "tier2_strong",
    "tier3_bubble",
    "tier4_bottom"
]

train_df_model["rank_bucket"] = pd.Categorical(
    train_df_model["rank_bucket"],
    categories=rank_categories,
    ordered=True
)

rank_dummies = pd.get_dummies(
    train_df_model["rank_bucket"], prefix="rank"
)
conf_dummies = pd.get_dummies(
    train_df_model["conference"], prefix="conf"
)

X = pd.concat([X, rank_dummies, conf_dummies], axis=1)

# 🔒 CANONICAL FEATURE LIST
FEATURE_COLS = X.columns.tolist()

# =====================================================
# 5. TRAIN / TEST SPLIT (ROW-LEVEL, DAILY SNAPSHOTS)
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =====================================================
# 6. XGBOOST PIPELINE (FINAL MODEL)
# =====================================================

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", XGBClassifier(
            n_estimators=500,
            max_depth=5,
            learning_rate=0.01,
            subsample=0.8,
            colsample_bytree=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
        )),
    ]
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("\n✅ XGBOOST PIPELINE REPORT\n")
print(classification_report(y_test, y_pred))

# =====================================================
# 7. SAVE MODEL
# =====================================================


MODEL_VERSION = "wc_field_pressure_v3"

dump(
    {
        "pipeline": pipeline,
        "features": FEATURE_COLS,
        "trained_on": pd.Timestamp.utcnow(),
        "model_type": "XGB-playoff-pipeline",
        "snapshot": "daily",
        "version": MODEL_VERSION,
    },
    f"models/playoff_xgb_pipeline_{MODEL_VERSION}.joblib"
)

print("💾 Model saved: models/playoff_xgb_pipeline_v3.joblib")

# =====================================================
# 8. BUILD DAILY PREDICTION MATRIX
# =====================================================

X_pred = (
    train_df_model
    .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
    .select_dtypes(include="number")
    .fillna(0)
)

rank_dummies = pd.get_dummies(
    train_df_model["rank_bucket"], prefix="rank"
)
conf_dummies = pd.get_dummies(
    train_df_model["conference"], prefix="conf"
)

X_pred = pd.concat([X_pred, rank_dummies, conf_dummies], axis=1)

X_pred = X_pred.reindex(columns=FEATURE_COLS, fill_value=0)

assert list(X_pred.columns) == FEATURE_COLS
print("✅ Feature alignment locked")

# =====================================================
# 9. DAILY PLAYOFF PROBABILITIES
# =====================================================

train_df_model["playoff_prob_final"] = (
    pipeline.predict_proba(X_pred)[:, 1]
)

# =====================================================
# 10. OPTIONAL TIERS
# =====================================================

def prob_tier(p):
    if p >= 0.85:
        return "Lock"
    elif p >= 0.65:
        return "Likely"
    elif p >= 0.45:
        return "Bubble"
    elif p >= 0.25:
        return "Danger"
    else:
        return "Long Shot"

train_df_model["playoff_tier"] = (
    train_df_model["playoff_prob_final"].apply(prob_tier)
)

print("\n✅ COMPLETE — DAILY GAME-BY-GAME PLAYOFF ODDS READY")


print(
    "Brier score:",
    brier_score_loss(y_test, y_prob)
)
# =====================================================
# SEASON PHASE BUCKETS
# =====================================================

def season_bucket(gp):
    if gp <= 20:
        return "early"
    elif gp <= 50:
        return "mid"
    else:
        return "late"

train_df_model["season_bucket"] = (
    train_df_model["games_played"].apply(season_bucket)
)

# =====================================================
# PERMUTATION IMPORTANCE BY SEASON PHASE (PIPELINE)
# =====================================================

bucket_importances = {}

for bucket in ["early", "mid", "late"]:
    mask = train_df_model["season_bucket"] == bucket

    X_bucket = X.loc[mask]
    y_bucket = y.loc[mask]

    # Safety check
    if len(X_bucket) < 200:
        print(f"⚠️ Skipping {bucket}: insufficient samples")
        continue

    perm = permutation_importance(
        pipeline,              # ✅ PIPELINE, not xgb_best
        X_bucket,
        y_bucket,
        n_repeats=10,
        random_state=42,
        n_jobs=-1,
        scoring="roc_auc",     # recommended for prob models
    )

    bucket_importances[bucket] = (
        pd.Series(
            perm.importances_mean,
            index=FEATURE_COLS
        )
        .sort_values(ascending=False)
    )

    print(f"✅ Permutation importance computed for: {bucket}")
for bucket, imp in bucket_importances.items():
    print(f"\n🔹 {bucket.upper()} SEASON")
    print(imp.head(10))
#store for re-use
importance_df = (
    pd.concat(bucket_importances, axis=1)
    .rename_axis("feature")
    .reset_index()
)
importance_df.head()



✅ XGBOOST PIPELINE REPORT

              precision    recall  f1-score   support

           0       0.92      0.91      0.91      2908
           1       0.90      0.92      0.91      2750

    accuracy                           0.91      5658
   macro avg       0.91      0.91      0.91      5658
weighted avg       0.91      0.91      0.91      5658

💾 Model saved: models/playoff_xgb_pipeline_v3.joblib
✅ Feature alignment locked

✅ COMPLETE — DAILY GAME-BY-GAME PLAYOFF ODDS READY
Brier score: 0.0663256917449629
✅ Permutation importance computed for: early
✅ Permutation importance computed for: mid
✅ Permutation importance computed for: late

🔹 EARLY SEASON
conference_rank        0.106402
sos_future             0.031294
c_shot_pct             0.012911
goal_diff              0.011154
corsi_diff_per_game    0.010873
ppng_rank              0.010860
division_rank          0.009198
SCF_pg                 0.009180
rush_for_per_game      0.006882
pk_rank                0.006402
dtype: float6

,feature,early,mid,late
0,conference_rank,0.106402,0.091856,0.018012
1,sos_future,0.031294,0.007655,0.001109
2,c_shot_pct,0.012911,0.007148,0.002432
3,goal_diff,0.011154,0.031199,0.027988
4,corsi_diff_per_game,0.010873,0.002652,0.000561
